# Programming Assignment 1: Manifold geometry for robot poses

**DD2603 Geometric Robot Learning, take-home assignment 1.**

A robot's end-effector pose is a position *and* an orientation. Positions live in a vector space, where you can add, subtract and average. Orientations do not. The set of rotations is a curved space, a **manifold**, and the usual vector operations either do not exist there or depend on how you write rotations down (Euler angles, quaternions, matrices).

In this assignment you implement the basic operations of Riemannian geometry yourself, in NumPy, and use them on real and synthetic robot data:

| Part | Topic | Data | Points |
|---|---|---|---|
| 0 | Setup and conventions | screwdriving | 3 |
| 1 | From Euclidean space to the sphere $S^2$: exp, log, distance, geodesics, parallel transport | synthetic, pouring | 21 |
| 2 | Injectivity radius and cut locus | synthetic | 11 |
| 3 | Orientation representations: quaternions, matrices, Euler angles | screwdriving | 18 |
| 4 | Robot poses on $\mathbb R^3\times S^3$ and trajectory interpolation | screwdriving | 18 |
| 5 | Riemannian vs Euclidean tracking-error metrics | screwdriving, pouring | 13 |
| – | Final synthesis | – | 16 |

### What you should be able to do after this assignment
1. Explain what a tangent space, the exponential and logarithm maps, geodesics and parallel transport are, and implement them on spheres.
2. Recognise where these maps stop being well defined (injectivity radius, cut locus) and what that does to computations.
3. Move between rotation matrices, unit quaternions and Euler angles, and explain the double cover $S^3\to SO(3)$ and the singularities of Euler angles.
4. Interpolate robot poses along geodesics, and compare with SLERP and with naive coordinate interpolation.
5. Choose and justify an error metric for orientation and pose tracking.
6. **Separate geometry from representation.** For every result, say whether it is **intrinsic** (it depends only on the rotations or poses), **representation-dependent** (Euler convention, quaternion sign) or **metric-dependent** (for example the weight $\lambda$ between metres and radians). This thread runs through the whole assignment.

## How the assignment works

Every part has the same structure, so you always know what is expected:

1. **Context:** a short recap of the theory, with the formulas you need.
2. **Tasks:** a numbered list naming each method or function to write, with input and output shapes.
3. **Skeleton:** code cells with signatures and docstrings. Replace each `raise NotImplementedError` with your code. Do not rename methods or change signatures: the checks call them directly.
4. **Public checks:** a provided cell, `checks.partN(...)`, that tests your code and prints `PASS`/`FAIL`/`TODO` per method with the largest error. Run it as often as you like.
5. **Experiments:** plotting and data loading are provided. The computations call *your* functions. Some cells contain small `TODO`s.
6. **Questions:** each question has an answer cell directly below it, starting with **Answer Qx.y**. Write your answer there.

### Rules
* **Your own classes and functions must not use geomstats.** geomstats is an independent *oracle* to compare against. You may call it only in cells that start with `# ORACLE`.
* NumPy and SciPy are allowed everywhere, except that you must not use `scipy.spatial.transform` inside the methods you implement. The provided wrappers in `assignment1utils.rotations` use it, and you can call those wrappers where the skeleton says so.
* Generative AI tools: follow the course policy. Answers must be tied to *your own* output (see below).

### What a full-mark written answer contains
1. The claim, in one or two sentences.
2. **A number or figure from your own run** that supports it (for example: "the largest Euler step at δ = 0 is 180°, against a true rate of 0.58°").
3. Where relevant, the label **intrinsic**, **representation-dependent** or **metric-dependent**.

An answer that is correct but not tied to your own output gets half marks.

### Grading
Each part is graded on your implementation (the public checks pass and the code is readable), the required figures and tables, and the written answers.

### Submission
Submit this notebook, executed, with all outputs (`.ipynb`). Before submitting, run *Runtime → Restart session and run all* and make sure that every cell runs without errors.

## The provided code: `assignment1utils`

The first code cell installs a small package, `assignment1utils`, which contains the data, conversion functions, plotting helpers and the public checks. You do not need to read its source, but everything is documented: run `help(assignment1utils.plotting)` or `assignment1utils.rotations.quat_to_matrix?` to see the docstrings.

* **`assignment1utils.data`** (all parts): `load_screw()`, `load_screw_tracked()`, `load_pouring()`. Each returns an object whose fields are NumPy arrays (`screw.poses`, `screw.delta_deg`, ...).
* **`assignment1utils.rotations`** (Parts 3–5): batched conversions, with quaternions in `(w, x, y, z)` order:
  * `quat_from_matrix`, `quat_to_matrix`, `quat_multiply`;
  * `euler_from_matrix`, `matrix_from_euler`, `wrap_angle`;
  * `rotvec_from_matrix` (the SO(3) log), `matrix_from_rotvec` (the SO(3) exp), `project_SO3` (SVD projection);
  * `make_pose`, `is_SE3`.
* **`assignment1utils.plotting`** (all parts): `plot_frames_3d` (path and RGB tool frames), `sphere_axes`, `plot_on_sphere`, `plot_tangent`, `plot_geodesic_fan`, `plot_frame_axes`, `plot_quat_components`, `plot_corr_matrix`, `subsample_idx`, and the fixed colours `METHOD_COLORS`. Every function returns its Matplotlib axes.
* **`assignment1utils.checks`** (end of every implementation step): `part1(Sphere)` ... `part5(metrics)`, the public checks.
* **`assignment1utils.oracle`** (Parts 1 and 3): geomstats, imported with the NumPy backend: `oracle.Hypersphere`, `oracle.SpecialOrthogonal`. **Only in `# ORACLE` cells.**

### The datasets

* **`load_screw()`**: `poses (3, 2501, 4, 4)`, `delta_deg = [3, 1, 0]`, `dt = 0.02`.
  *Synthetic screwdriving.* The tool turns 4 full revolutions about its own z-axis at constant speed (0.576° per frame, 50 Hz) while it advances 32 mm along that axis. The goal orientation is a rotation of 90° + δ about the world y-axis, so it sits δ degrees from the singularity of yaw–pitch–roll Euler angles.
* **`load_screw_tracked()`**: `reference`, `tracked (3, 2501, 4, 4)`, `offset_rotvec (3,)`, `offset_pos (3,)`.
  The same demos and a "tracked" copy, `tracked = reference @ T_off`, with a constant tool-frame offset of 1° and 2 mm. The true tracking error is known and constant.
* **`load_pouring()`**: `water`, `wine (5, 480, 4, 4)`, `position_names`.
  Demonstrations of pouring water or wine from a bottle (the pose of the bottle) at 5 target positions, from very left to very right. Water and wine demos for the same position are similar but not identical.

### Your API: what you implement

| Part | You implement |
|---|---|
| 1 | `Sphere`: the basic geometry of the sphere |
| 2 | two small functions to explore the cut locus |
| 3 | `UnitQuaternions`: quaternion signs and rotation distances |
| 4 | `PoseManifold`: geometry of robot poses, and pose interpolation |
| 5 | six tracking-error metrics |

Later parts build on earlier ones: `UnitQuaternions` inherits your `Sphere` code, and `PoseManifold` uses `UnitQuaternions`. Make the public checks of a part pass before you move on.

## Conventions (read this once, carefully)

* **Quaternions are `(w, x, y, z)`, scalar first.** SciPy uses `(x, y, z, w)`, and the wrappers in `assignment1utils.rotations` reorder for you. A public check catches a swapped order.
* **Euler angles use SciPy's `"ZYX"` sequence** (`rotations.EULER_SEQ`). In SciPy, **uppercase letters mean intrinsic rotations** (about the moving axes) and lowercase letters mean extrinsic rotations (about the fixed axes). `"ZYX"` is yaw–pitch–roll: $R = R_z(\text{yaw})\,R_y(\text{pitch})\,R_x(\text{roll})$. It is singular when the pitch is ±90°, and the screwdriving data is built to pass through (δ = 0) or close to that singularity.
* **Shapes.** Every method accepts a single element, for example a point of shape `(3,)`, or a batch of shape `(N, 3)`, and returns the matching shape. Broadcasting a single base point against a batch must work. Batched code is what makes the experiments on 2501-frame trajectories fast.
* **The base point always comes first** in our API: `S.log(p, q)` is $\mathrm{Log}_p(q)$. geomstats uses the opposite order (see Part 1).
* **Angles are radians inside the API.** Plots and printed tables use degrees, and say so.
* **Numerical accuracy.** Your maps must be exact up to rounding (the checks allow errors of about $10^{-8}$; a good implementation reaches $10^{-14}$), must not return `NaN` when an input is exactly zero, and must stay accurate for very small angles. Part 1 explains what this requires, and the checks test it.
* **Poses** are 4×4 homogeneous matrices $T = \begin{pmatrix} R & p\\ 0 & 1\end{pmatrix}$, in metres. The columns of $R$ are the tool's x-, y- and z-axes written in world coordinates. Part 0 explains why this format is used.

---
# Part 0: Setup and conventions (3 points)

Run the next cell first, and run it again every time Colab restarts the runtime. It installs `assignment1utils` (with the data) and geomstats if they are missing, and prints the versions. If you work locally and have installed `assignment1utils` yourself (see the README of this assignment on GitHub), it skips the installation. If something fails, include a screenshot of this cell's output when you ask for help.

In [ ]:
# Part 0 setup: safe to re-run.
GITHUB_REPO = "GeoRob-Lab/dd2603-course"   # the course repository on GitHub (set by the TA)
ASSIGNMENT_RELEASE = "assignment1-v1.0"                 # the release of this assignment (a git tag)

import importlib.util
if importlib.util.find_spec("assignment1utils") is None:     # not installed yet, e.g. a fresh Colab runtime
    %pip install -q "assignment1utils @ git+https://github.com/{GITHUB_REPO}@{ASSIGNMENT_RELEASE}#subdirectory=assignments/assignment1"

import numpy as np
import matplotlib.pyplot as plt
import assignment1utils
from assignment1utils import data, checks
from assignment1utils import rotations as rot
from assignment1utils import plotting as lp

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
np.set_printoptions(precision=4, suppress=True)
assignment1utils.print_versions()

All randomness in the assignment goes through `rng`. You may change `SEED`; your synthetic data, and therefore some of your numbers, depend on it.

In [ ]:
SEED = 2603
rng = np.random.default_rng(SEED)

### Poses as 4×4 matrices

All datasets store an end-effector pose as a **homogeneous transformation**
$$T=\begin{pmatrix}R & p\\ 0 & 1\end{pmatrix}\in SE(3),\qquad R\in SO(3),\ p\in\mathbb R^3 .$$
$p$ is the position of the tool origin in the world frame, and the columns of $R$ are the tool's x-, y- and z-axes written in world coordinates.

Why 16 numbers for a 6-dimensional object? Because rigid motions then become matrix algebra:
* **Moving points.** A point with tool coordinates $x$ has world coordinates $Rx+p$, which is one matrix product with the homogeneous vector: $T\begin{pmatrix}x\\1\end{pmatrix}=\begin{pmatrix}Rx+p\\1\end{pmatrix}$.
* **Chaining frames.** If $T_{WA}$ is the pose of frame A in the world and $T_{AB}$ the pose of B in A, then $T_{WB}=T_{WA}T_{AB}$. Kinematic chains (base → links → tool → object) are products of 4×4 matrices. Multiplying on the **right** applies a motion expressed in the tool frame; multiplying on the **left** applies it in the world frame.
* **Inverting.** $T^{-1}=\begin{pmatrix}R^\top & -R^\top p\\ 0 & 1\end{pmatrix}$, cheap and exact.
* **No singularities and no ambiguity.** Every pose has exactly one matrix, unlike Euler angles (singular at some angles) or quaternions (two per rotation). This is why robot software (forward kinematics, ROS `tf`) and these datasets use it, and why we use it as the reference format from which we compute everything else.

The price is redundancy: the 9 entries of $R$ must satisfy $R^\top R=I$ and $\det R=1$. Operations that ignore these constraints, such as averaging or linearly interpolating the entries, leave $SO(3)$ (Part 4).

### The screwdriving dataset

`data.load_screw()` returns three **synthetic** demonstrations of a screwing motion: the tool turns about its own z-axis at constant speed while it advances along that axis, like a screwdriver driving a screw. It was generated analytically, so the true motion is known exactly:

1. **Goal pose.** Position $(0.55, 0.10, 0.45)$ m. Orientation $R_\text{goal}=R_y(90°+\delta)$, a rotation about the world y-axis. The tool z-axis, i.e. the screw axis, is therefore $R_\text{goal}e_z=(\cos\delta,\,0,\,-\sin\delta)$: almost the world x-axis, tilted down by δ.
2. **Twist.** A constant twist in the tool frame: 4 full turns about the tool z-axis ($4\cdot360°=1440°$) and an advance of 8 mm per turn along it (32 mm in total).
3. **Trajectory.** $T(s)=T_\text{start}\exp(s\,\xi)$ for $s\in[0,1]$, with $T_\text{start}=T_\text{goal}\exp(-\xi)$ so that the motion ends exactly at the goal. In terms of rotations, $R(t)=R_\text{start}R_z(\theta(t))$, where $\theta$ grows by **0.576° per frame**.
4. **Sampling.** 0.08 revolutions per second for 50 s at 50 Hz: **2501 frames** per demo, one revolution every 625 frames (12.5 s). The slow speed puts many frames near the special orientations discussed below.

The three demos differ only in **δ = 3°, 1°, 0°**. As the tool spins, the pitch of its yaw–pitch–roll Euler angles swings between $-(90°-\delta)$ and $+(90°-\delta)$ twice per revolution: with δ = 0 it reaches ±90°, the singular configuration of that chart, and with δ = 1° and 3° it passes within 1° and 3° of it (Part 3). The motion is the simplest one possible (a rotation about a fixed axis at constant speed), so whatever goes wrong with it later in the assignment is caused by the representation, not by the motion.

### A first look at the data

The figure shows the δ = 1° demo (the three demos look identical at this scale):
* **(a)** the path (a 32 mm straight line) with tool frames;
* **(b)** the tool frame at 7 instants of the first half revolution, every 30°, drawn as arrows from the origin to the unit sphere (later instants are more opaque): the blue z-axis stays fixed, and the red x- and green y-axes turn about it.

In [ ]:
screw = data.load_screw()
print("poses", screw.poses.shape, "| delta_deg", screw.delta_deg, "| dt", screw.dt, "s")
print("valid SE(3):", rot.is_SE3(screw.poses, atol=1e-12))

T = screw.poses[1]                                    # the δ = 1° demo, (2501, 4, 4)
Rs = T[:, :3, :3]
t_s = np.arange(len(T)) * screw.dt                    # time [s]
angle_to_start = np.degrees(np.linalg.norm(rot.rotvec_from_matrix(Rs[0].T @ Rs), axis=-1))
step = np.degrees(np.linalg.norm(rot.rotvec_from_matrix(np.swapaxes(Rs[:-1], -1, -2) @ Rs[1:]), axis=-1))
accumulated = np.concatenate([[0.0], np.cumsum(step)])

fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
lp.plot_frames_3d(T, ax=ax, n_frames=40, frame_len=0.004, colors=["k"], title="(a) Path and tool frames")
ax = fig.add_subplot(1, 2, 2, projection="3d")
lp.sphere_axes(ax, elev=15, azim=-35)
idx = np.arange(7) * 52                               # every 52 frames = 30°
lp.plot_frame_axes(Rs[idx], ax=ax, labels=[f"{30 * k}°" for k in range(7)],
                   title="(b) Tool frame during the first half turn")
plt.tight_layout(); plt.show()

print(f"rotation per frame: {step.min():.4f}..{step.max():.4f} deg, accumulated: {accumulated[-1]:.1f} deg, "
      f"angle between first and last frame: {angle_to_start[-1]:.2e} deg")

### Questions, Part 0
**Q0.1** Given a pose matrix `T` (shape `(4, 4)`), where are the tool position and the tool z-axis? Give the NumPy indexing and say in which frame the z-axis is expressed.

**Answer Q0.1:**

_Your answer here._

**Q0.2** Each screwdriving demo turns 4 full revolutions. Compute the rotation angle between the first and the last frame of a demo in the cell below. What is it, and what does it imply for interpolating from the start pose to the end pose?

In [ ]:
# TODO: rotation angle (deg) between the first and last frame of each demo.
# Hint: rot.rotvec_from_matrix(R0.T @ R1) is a rotation vector; its norm is the angle.

**Answer Q0.2:**

_Your answer here._

---
# Part 1: From Euclidean space to the sphere $S^2$ (21 points)

**Why this part.** Everything later in the assignment is built from a handful of operations: the exponential and logarithm maps, geodesics, distances and parallel transport. Interpolating poses (Part 4) walks along geodesics, and error metrics (Part 5) are distances; in Programming Assignment 2, sampling pushes tangent vectors through Exp and averaging averages Log vectors. Here you implement these operations on the sphere, where you can still draw everything, and the experiments show what curvature changes compared with $\mathbb R^n$. Each experiment prepares one of the questions at the end of the part.

### Context
The unit sphere $S^n=\{x\in\mathbb R^{n+1}:\lVert x\rVert=1\}$ is the simplest curved manifold, and it is the one we need: $S^2$ holds directions (for example a tool axis), and $S^3$ holds unit quaternions (Part 3). We work with it **extrinsically**: points are unit vectors in $\mathbb R^{n+1}$.

**Tangent space.** At a point $p$, the tangent space is $T_pS^n=\{v\in\mathbb R^{n+1}:\langle p,v\rangle=0\}$, the plane that touches the sphere at $p$. It is a *vector space*: you can add tangent vectors, scale them and average them. The orthogonal projection of any $v\in\mathbb R^{n+1}$ onto it is $v-\langle p,v\rangle p$.

**Metric.** We use the metric induced by $\mathbb R^{n+1}$: $\langle u,v\rangle_p=u^\top v$ for $u,v\in T_pS^n$.

**Geodesics** are the great circles. The geodesic that leaves $p$ with velocity $v$ is $\gamma(t)=\cos(t\lVert v\rVert)\,p+\sin(t\lVert v\rVert)\,v/\lVert v\rVert$.

**Exponential map.** $\mathrm{Exp}_p(v)=\gamma(1)$: walk along the geodesic with initial velocity $v$ for unit time, which covers arc length $\lVert v\rVert$:
$$\mathrm{Exp}_p(v)=\cos(\lVert v\rVert)\,p+\frac{\sin\lVert v\rVert}{\lVert v\rVert}\,v .$$

**Logarithm map.** $\mathrm{Log}_p(q)$ is the inverse: the tangent vector at $p$ that points along the shortest great-circle arc to $q$, with length equal to that arc. **You derive it below** (Task 1.0).

**Distance.** $d(p,q)=\lVert\mathrm{Log}_p q\rVert=\theta$, the angle between $p$ and $q$.

**Parallel transport** moves a tangent vector from $T_pS^n$ to $T_qS^n$ along the geodesic without "turning" it, so that lengths and angles are kept. Along the minimising geodesic from $p$ to $q$ (with $\theta=d(p,q)$):
$$\Gamma_{p\to q}(v)=v-\frac{\langle \mathrm{Log}_p q,\,v\rangle}{\theta^2}\bigl(\mathrm{Log}_p q+\mathrm{Log}_q p\bigr)
\;=\;v-\frac{\langle q,v\rangle}{1+\langle p,q\rangle}\,(p+q).$$
The two forms are equal. The second one has no $0/0$ when $p=q$.

### Numerics that the checks test
The formulas above are exact, but three details decide whether code based on them works in practice. The skeleton reminds you of each one where it matters.

1. **Exactly zero inputs.** $\mathrm{Exp}_p(0)$ must return $p$, and $\mathrm{Log}_p(p)$ must return $0$. Evaluate the formulas at $\lVert v\rVert=0$ and at $\theta=0$: what does NumPy compute there, and what should the value be? This is not a corner case, since the first point of every geodesic is $\mathrm{Exp}_p(0)$. Remember that `np.where(cond, a, b)` evaluates both `a` and `b`.
2. **Small angles.** $\theta=\arccos\langle p,q\rangle$ is correct on paper but inaccurate in floating point when θ is small (look at the slope of arccos near 1). A check measures the distance between points $10^{-7}$ rad apart. Hint: $\sin\theta$ can also be computed from $p$ and $q$; then look at `np.arctan2`.
3. **The antipode.** $\mathrm{Log}_p(-p)$ is not defined (Part 2 shows why). Do not try to handle it, but your code must return finite numbers for points near $-p$.

### Tasks
* **1.0** In the markdown cell below, derive $\mathrm{Log}_p(q)$ by inverting the formula for $\mathrm{Exp}_p$. Hint: write $q=\cos\theta\,p+\sin\theta\,u$ with $u\in T_pS^n$ a unit vector.
* **1.1** `belongs(x)`, `projection(x)`, `to_tangent(p, v)` and `is_tangent(p, v)`, which tests $|\langle p,v\rangle|\le$ `atol` $\cdot\max(1,\lVert v\rVert)$.
* **1.2** `inner(p, u, v)` and `norm(p, v)`. The Riemannian inner product is only defined for tangent vectors at $p$, so `inner` must raise a `ValueError` if `u` or `v` is not tangent at `p` (use `is_tangent`).
* **1.3** `exp(p, v)`, `log(p, q)` and `dist(p, q)`, including exactly zero inputs.
* **1.4** `geodesic(p, q, t)`: for an array `t` of shape `(T,)`, return the points $\mathrm{Exp}_p(t_i\,\mathrm{Log}_p q)$, shape `(T,) + p.shape`.
* **1.5** `parallel_transport(p, q, v)`.
* **1.6** `random_tangent(p, scale, rng)`: a random Gaussian tangent vector at each `p`.

All methods take NumPy arrays of shape `(n+1,)` or `(N, n+1)` and must broadcast a single `p` against a batch.

**Task 1.0: derivation of $\mathrm{Log}_p(q)$.**

**Answer 1.0:**

_Your answer here._

In [ ]:
class Sphere:
    """
    The unit sphere S^n = {x in R^(n+1) : |x| = 1}, with the metric induced by R^(n+1).

    Points and tangent vectors are NumPy arrays of shape (n+1,) or (..., n+1).
    All methods broadcast over the leading dimensions.
    The base point is always the FIRST argument (geomstats uses the opposite order).
    """

    def __init__(self, n=2):
        self.n = n                # intrinsic dimension
        self.ambient_dim = n + 1  # points live in R^(n+1)

    def __repr__(self):
        return f"{type(self).__name__}(n={self.n})"

    def random_point(self, n_samples=None, rng=None):
        """Uniformly distributed point(s): shape (n+1,) or (n_samples, n+1). Provided."""
        rng = np.random.default_rng() if rng is None else rng
        shape = (self.ambient_dim,) if n_samples is None else (n_samples, self.ambient_dim)
        x = rng.normal(size=shape)
        return x / np.linalg.norm(x, axis=-1, keepdims=True)

    # ------------------------------------------------------------ Task 1.1
    def belongs(self, x, atol=1e-10):
        """Boolean array (...,): True where | |x| - 1 | < atol."""
        raise NotImplementedError

    def projection(self, x):
        """Closest point on the sphere to x != 0 (radial projection). (..., n+1) -> (..., n+1)."""
        raise NotImplementedError

    def to_tangent(self, p, v):
        """Orthogonal projection of the ambient vector v onto T_p S^n."""
        raise NotImplementedError

    def is_tangent(self, p, v, atol=1e-8):
        """Boolean array (...,): True where v is in T_p S^n, i.e. |<p, v>| <= atol * max(1, |v|)."""
        raise NotImplementedError

    # ------------------------------------------------------------ Task 1.2
    def inner(self, p, u, v):
        """
        Riemannian inner product <u, v>_p of tangent vectors at p. Returns shape (...).
        Raise ValueError if u or v is not tangent at p (use is_tangent).
        """
        raise NotImplementedError

    def norm(self, p, v):
        """Riemannian norm |v|_p of a tangent vector at p. Returns shape (...)."""
        raise NotImplementedError

    # ------------------------------------------------------------ Task 1.3
    def exp(self, p, v):
        """
        Exponential map Exp_p(v) for v in T_p S^n.
        Numerics: v can be exactly 0 (e.g. the first point of a geodesic), and then
        sin|v| / |v| is 0/0. Exp_p(0) must return p, not NaN (Numerics, point 1).
        """
        raise NotImplementedError

    def log(self, p, q):
        """
        Logarithm map Log_p(q) in T_p S^n.
        Numerics: (1) q == p must give 0, not NaN; (2) compute the angle with atan2, not
        arccos, or small angles are inaccurate; (3) Log_p(-p) is undefined (every direction
        reaches -p): do not special-case it, but return finite values for q near -p.
        """
        raise NotImplementedError

    def dist(self, p, q):
        """Geodesic distance (the angle between p and q), shape (...). Use atan2, not arccos (Numerics, point 2)."""
        raise NotImplementedError

    # ------------------------------------------------------------ Task 1.4
    def geodesic(self, p, q, t):
        """
        Points Exp_p(t_i Log_p q) on the minimising geodesic from p (t=0) to q (t=1).
        t: array-like of shape (T,) (a list is allowed). Returns shape (T,) + broadcast shape of p and q.
        """
        raise NotImplementedError

    # ------------------------------------------------------------ Task 1.5
    def parallel_transport(self, p, q, v):
        """
        Parallel transport of v in T_p S^n to T_q S^n along the minimising geodesic.
        Use the second formula of the context: it has no 0/0 when q == p.
        """
        raise NotImplementedError

    # ------------------------------------------------------------ Task 1.6
    def random_tangent(self, p, scale=1.0, rng=None):
        """A random tangent vector at each p: an isotropic Gaussian with std `scale`, projected to T_p."""
        rng = np.random.default_rng() if rng is None else rng
        raise NotImplementedError


S2 = Sphere(2)
S2

### Public checks
The checks first test identities ($\mathrm{Exp}\circ\mathrm{Log}=\mathrm{id}$, constant speed of geodesics, the triangle inequality, and so on) and then compare each method with geomstats' `Hypersphere(dim=2)` on 1000 random inputs. The log comparison leaves out pairs within $10^{-3}$ rad of antipodal, where Log is ill-conditioned.

In [ ]:
checks.part1(Sphere)

### Using geomstats
Geomstats is useful to check things the public checks do not cover. Its `log` takes the **base point second**: `metric.log(point, base_point)`. Fill in the call below so that it computes $\mathrm{Log}_p(q)$, and confirm that it matches your `S2.log(p, q)`.

In [ ]:
# ORACLE: geomstats is allowed in this cell only.
from assignment1utils import oracle
G = oracle.Hypersphere(dim=2)
p, q = S2.random_point(rng=rng), S2.random_point(rng=rng)
gs_log = G.metric.log(q, p)
print("ours      ", S2.log(p, q))
print("geomstats ", gs_log)
print("max difference:", np.abs(S2.log(p, q) - gs_log).max())

### Experiment 1.1: chord versus arc
For two points at angle θ, compare the Euclidean (chord) distance $\lVert p-q\rVert$ in $\mathbb R^3$ with the geodesic distance $d(p,q)$. Build the points with your own `exp` and measure with your own `dist`.

**Plot** a figure with two panels:
* left: both distances against the angle between the points (in degrees);
* right: the relative error of the chord, $1-\text{chord}/\text{arc}$, against the angle on **log–log** axes, together with the reference curve $\theta^2/24$. On log–log axes a power law $\theta^k$ is a straight line of slope $k$.

→ Q1.3 asks what the slope tells you.

In [ ]:
p = np.array([0.0, 0.0, 1.0])
e1 = np.array([1.0, 0.0, 0.0])
theta = np.linspace(1e-3, np.pi, 200)

# TODO: q[i] = Exp_p(theta[i] * e1); chord[i] = |p - q[i]|; arc[i] = dist(p, q[i])
rel_err = 1 - chord / arc

# TODO: the two-panel figure described above (x-axis in degrees; use ax.loglog for the right panel)
print("relative error of the chord: " + ", ".join(
    f"{np.interp(a, np.degrees(theta), rel_err):.1e} at {a}°" for a in [1, 10, 90]) + f", {rel_err[-1]:.2f} at 180°")

### Experiment 1.2: the straight line leaves the sphere
Take the straight segment $(1-t)p+tq$ between two points of the sphere, and its normalisation $\mathrm{nlerp}(t)=\frac{(1-t)p+tq}{\lVert(1-t)p+tq\rVert}$. nlerp is a cheap and popular alternative to the geodesic; on $S^3$ it is the "normalised lerp" of quaternions used in computer graphics.

**Plot**, for θ ∈ {30°, 90°, 150°, 179°}, a figure with two panels:
* left: the norm $\lVert(1-t)p+tq\rVert$ against $t$;
* right: the speed of nlerp against $t$ on a log scale, in degrees per unit $t$ (the geodesic distance between consecutive points divided by the step in $t$), with the speed of the geodesic as a dotted horizontal line of the same colour.

The second figure (provided) draws the three curves on the sphere for θ = 150°.

→ Q1.1 (the norm) and Q1.4 (the speed).

In [ ]:
t = np.linspace(0, 1, 401)
# TODO: for each angle, seg = (1-t) p + t q (shape (401, 3)), nlerp = seg normalised,
#       speed = dist between consecutive nlerp points / dt (shape (400,)); draw the two-panel figure

In [ ]:
# The same comparison on the sphere for θ = 150°, with markers at t = 0, 0.1, ..., 1. Provided.
q = S2.exp(p, np.radians(150) * e1)
tk = np.linspace(0, 1, 11)
geo = S2.geodesic(p, q, tk)                             # on the sphere, evenly spaced
seg = (1 - tk)[:, None] * p + tk[:, None] * q           # straight line, inside the ball
nl = S2.projection(seg)                                 # nlerp: the line pushed radially onto the sphere

ax = lp.sphere_axes(elev=0, azim=-90, figsize=(6, 6), title="θ = 150°, markers at t = 0, 0.1, ..., 1")
lp.plot_on_sphere(S2.geodesic(p, q, np.linspace(0, 1, 100)), ax=ax, kind="curve", color="C0", lw=1)
ax.plot(*geo.T, "o", color="C0", label="geodesic  Exp_p(t Log_p q)")
ax.plot(*seg.T, "s--", color="C3", ms=4, label="straight line  (1-t) p + t q")
ax.plot(*nl.T, "^", color="C1", ms=7, label="nlerp (normalised line)")
for a, b in zip(seg, nl):
    ax.plot(*np.stack([np.zeros(3), b]).T, color="0.6", lw=0.6)    # radial projection of the line
ax.scatter(0, 0, 0, color="k", s=10)
ax.set_yticks([]); ax.set_ylabel("")                     # we look along the y-axis
ax.legend(fontsize=7, loc="lower left"); plt.show()

### Experiment 1.3: holonomy, or why curvature is real
**Parallel transport around a loop.** Parallel transport carries a tangent vector along a curve without turning it: its length, and its angle with any other transported vector, stay fixed, and along a geodesic it keeps a constant angle with the direction of the geodesic. In the plane this is ordinary translation. Carry a vector around a closed loop and compare it with where it started: the angle between the two is the **holonomy** of the loop.

**Two facts you need.**
* *Gauss–Bonnet.* For a loop around a region $D$ of a surface with Gaussian curvature $K$, the holonomy angle is $\int_D K\,dA$.
* *Girard's theorem.* On the unit sphere, a geodesic triangle with interior angles α, β, γ has area α + β + γ − π, its **angle excess**. The interior angle at a corner $X$ is the angle between the initial directions of the two sides that leave $X$, that is between $\mathrm{Log}_XY$ and $\mathrm{Log}_XZ$.

**What you do.**
1. For the octant triangle $N=(0,0,1)$, $A=(1,0,0)$, $B=(0,1,0)$, transport $v_0=(1,0,0)\in T_NS^2$, which points along the first side, around $N\to A\to B\to N$, and measure the angle between $v_0$ and the vector that comes back. Compute the area of the triangle from its interior angles. The provided figure shows the transported vector at 8 points of each side, coloured from dark (start) to yellow (end). Follow the arrow around the corners, and compare how much the path turns with how much the arrow turns.
2. For the triangles $N$, $\mathrm{Exp}_N(a\,e_1)$, $\mathrm{Exp}_N(a\,e_2)$ with $a$ from 0.05 to π/2, compute the holonomy and the area. **Plot** the holonomy against the area, with the line $y=x$ for reference.

→ Q1.5.

In [ ]:
def transport_around(S, corners, v0):
    """Transport v0 (tangent at corners[0]) along the closed polygon corners[0] -> ... -> corners[0]."""
    # TODO
    raise NotImplementedError


def interior_angle(S, X, Y, Z):
    """Interior angle at X of the geodesic triangle XYZ: the angle between Log_X Y and Log_X Z."""
    # TODO
    raise NotImplementedError


def signed_angle(u, v, normal):
    """Signed angle (rad) from u to v, about `normal`. Provided."""
    return np.arctan2(np.dot(np.cross(u, v), normal), np.dot(u, v))


def transport_path(S, corners, v0, n_per_side=8):
    """Points along the closed polygon and the transported vector at each of them. Provided."""
    pts, vecs, v = [], [], v0
    for a, b in zip(corners, list(corners[1:]) + [corners[0]]):
        for s in np.linspace(0, 1, n_per_side, endpoint=False):
            x = S.geodesic(a, b, [s])[0]
            pts.append(x)
            vecs.append(S.parallel_transport(a, x, v))
        v = S.parallel_transport(a, b, v)
    return np.array(pts), np.array(vecs)

The octant triangle

In [ ]:
N, A, B = np.eye(3)[2], np.eye(3)[0], np.eye(3)[1]
v0 = np.array([1.0, 0.0, 0.0])        # tangent at N, pointing along the side N -> A
v1 = transport_around(S2, [N, A, B], v0)
angles = [interior_angle(S2, N, A, B), interior_angle(S2, A, B, N), interior_angle(S2, B, N, A)]
print(f"interior angles: {np.degrees(angles).round(4)} deg -> area (angle excess) = {sum(angles) - np.pi:.6f} sr")
print(f"after the loop v = {v1.round(6)}: rotated by {np.degrees(signed_angle(v0, v1, N)):.4f} deg "
      f"= {signed_angle(v0, v1, N):.6f} rad")

pts, vecs = transport_path(S2, [N, A, B], v0)
ax = lp.sphere_axes(elev=30, azim=40, figsize=(6.5, 6.5), title="Parallel transport of $v_0$ around N → A → B → N")
for a, b in [(N, A), (A, B), (B, N)]:
    lp.plot_on_sphere(S2.geodesic(a, b, np.linspace(0, 1, 50)), ax=ax, kind="curve", color="k", lw=1.5)
for x, v, c in zip(pts, vecs, plt.cm.viridis(np.linspace(0, 1, len(pts)))):
    lp.plot_tangent(x, v, ax=ax, scale=0.25, color=c)
lp.plot_tangent(N, v0, ax=ax, scale=0.45, color="C0", lw=3, label="$v_0$ (start)")
lp.plot_tangent(N, v1, ax=ax, scale=0.45, color="C3", lw=3, label="after the loop")
for P, name in [(N, "N"), (A, "A"), (B, "B")]:
    ax.text(*(1.12 * P), name, fontsize=11)
ax.legend(fontsize=8); plt.show()

A family of triangles: holonomy against area

In [ ]:
a_vals = np.linspace(0.05, np.pi / 2, 25)
holonomy, area = [], []
for a in a_vals:
    X, Y = S2.exp(N, a * A), S2.exp(N, a * B)
    v = transport_around(S2, [N, X, Y], v0)
    holonomy.append(abs(signed_angle(v0, v, N)))
    area.append(interior_angle(S2, N, X, Y) + interior_angle(S2, X, Y, N) + interior_angle(S2, Y, N, X) - np.pi)

# TODO: plot the holonomy against the area (markers), with the line y = x for reference

### Experiment 1.4: a real $S^2$-valued signal
The direction of the bottle's z-axis during a pouring demonstration is a curve on $S^2$. Compute the length of this curve (the sum of the geodesic distances between consecutive frames) and the geodesic distance between its first and last point. **Plot** the curve on the sphere, together with the geodesic from its first to its last point (use `lp.sphere_axes` and `lp.plot_on_sphere(..., kind="curve")`).

→ Q1.6.

In [ ]:
pour = data.load_pouring()
z = pour.water[0][:, :3, 2]                      # bottle z-axis over time, (480, 3)
# TODO: path_len = sum of dist between consecutive points; direct = dist(z[0], z[-1])
print(f"length of the z-axis curve: {np.degrees(path_len):.1f} deg; geodesic start -> end: {np.degrees(direct):.1f} deg")

# TODO: plot the curve z on the sphere and the geodesic from z[0] to z[-1]

### Questions, Part 1
**Q1.1** Why does the straight segment between two points of the sphere leave the sphere? Give $\lVert(1-t)p+tq\rVert^2$ as a function of $t$ and $\theta$, and check it against the left panel of Experiment 1.2 (for example its minimum for θ = 90°).

**Answer Q1.1:**

_Your answer here._

**Q1.2** Why does `log` return a tangent vector and not a point? What do its direction and its length mean?

**Answer Q1.2:**

_Your answer here._

**Q1.3** What slope do you read in the right panel of Experiment 1.1, and what does it say about replacing geodesic distances by Euclidean ones, for nearby and for distant points? Use this to explain why tangent spaces are useful for computation, and name two operations in this assignment that are carried out in a tangent space.

**Answer Q1.3:**

_Your answer here._

**Q1.4** nlerp and the geodesic trace the same curve on the sphere. When does the difference between them matter for a robot? Use the speed plot.

**Answer Q1.4:**

_Your answer here._

**Q1.5** What holonomy angle and what area did you measure for the octant triangle, and what would the holonomy be for a triangle in the plane? In the transport figure, how much does the path turn at the corners, and how much does the arrow turn? Use your second figure and Gauss–Bonnet to estimate the Gaussian curvature $K$ of the unit sphere.

**Answer Q1.5:**

_Your answer here._

**Q1.6** In Experiment 1.4, compare the length of the bottle's z-axis curve with the geodesic distance between its end points. What does the ratio tell you about the motion? Would interpolating along the geodesic from the first to the last orientation reproduce this demonstration?

**Answer Q1.6:**

_Your answer here._

**Checkpoint, Part 1.** All public checks pass, the figures above are shown, and Q1.1 to Q1.6 are answered.

---
# Part 2: Injectivity radius and cut locus (11 points)

### Context
The sphere is **complete**: $\mathrm{Exp}_p(v)$ is defined for every tangent vector $v$, however long (Hopf–Rinow). What fails for long vectors is that $\mathrm{Exp}_p$ stops being **injective** and the geodesics stop being **minimising**.

* The **cut locus** of $p$ is the set of points beyond which the geodesics from $p$ stop minimising. On $S^2$ it is the single antipode $-p$: every geodesic from $p$ reaches $-p$ after length π, and after that it comes back towards $p$ the other way round.
* The **injectivity radius** at $p$ is the largest $r$ such that $\mathrm{Exp}_p$ is a diffeomorphism from the open ball $\{\lVert v\rVert<r\}\subset T_pM$ onto its image. It is the distance from $p$ to its cut locus. For the unit sphere it is π everywhere.
* $\mathrm{Log}_p$ is the inverse of $\mathrm{Exp}_p$ *on that ball only*. For $\lVert v\rVert>\pi$, $\mathrm{Log}_p(\mathrm{Exp}_p v)$ returns the *short* way round, not $v$.

Statistics on manifolds (means and Gaussians, Programming Assignment 2) work in tangent spaces through Log. Data that reaches the cut locus therefore breaks them, which is why this part matters.

### Tasks
* **2.1** `exp_fan(S, p, radii, n_dirs)`: choose `n_dirs` unit tangent directions $u_i$ at $p$, evenly spaced in angle, and return the array `rays[i, j]` $=\mathrm{Exp}_p(r_j u_i)$ of shape `(n_dirs, len(radii), n+1)`. Use your `Sphere` methods.
* **2.2** `log_roundtrip_error(S, p, v)` $=\lVert\mathrm{Log}_p(\mathrm{Exp}_p v)-v\rVert$.

In [ ]:
def exp_fan(S, p, radii, n_dirs):
    """rays[i, j] = Exp_p(radii[j] * u_i) for n_dirs unit tangent directions u_i at p (2-sphere)."""
    # TODO: build an orthonormal basis (e1, e2) of T_p S^2, set u_i = cos(a_i) e1 + sin(a_i) e2.
    raise NotImplementedError


def log_roundtrip_error(S, p, v):
    """| Log_p(Exp_p v) - v |, shape (...)."""
    raise NotImplementedError


checks.part2(Sphere, exp_fan, log_roundtrip_error)

### Experiment 2.1: a fan of geodesics from the north pole
**Plot** three panels showing the rays from $p=(0,0,1)$ up to length π/2, π and 2π (60 radii and 16 directions each). Use 3D subplots, `lp.sphere_axes(ax)` and `lp.plot_geodesic_fan(p, rays, radii, ax=ax)`. Watch where the rays meet.

→ Q2.2.

In [ ]:
p = np.array([0.0, 0.0, 1.0])
# TODO: one 3D panel per maximum length r_max in (pi/2, pi, 2 pi): radii = np.linspace(0, r_max, 60),
#       rays = exp_fan(S2, p, radii, 16), then lp.sphere_axes(ax) and lp.plot_geodesic_fan(p, rays, radii, ax=ax)

*Optional, interactive.* The next cell draws the same fan with a slider for the maximum radius (it needs `ipywidgets`, which Colab has). Sliders do not survive the PDF export, so keep the static figure above.

In [ ]:
try:
    from ipywidgets import FloatSlider, interact

    def _fan(rmax_over_pi=1.0):
        radii = np.linspace(0, rmax_over_pi * np.pi, 60)
        lp.plot_geodesic_fan(p, exp_fan(S2, p, radii, 16), radii, ax=lp.sphere_axes(elev=15))
        plt.show()

    interact(_fan, rmax_over_pi=FloatSlider(min=0.1, max=2.0, step=0.05, value=1.0))
except ImportError:
    print("ipywidgets is not installed; skip this cell.")

### Experiments 2.2 and 2.3: distance and round-trip error along a ray
Walk along one ray, $v=r\,u$ with $r\in[0,3\pi]$. **Plot** two panels against $r/\pi$: the distance $d(p,\mathrm{Exp}_p(ru))/\pi$, and `log_roundtrip_error`$/\pi$. Mark $r=\pi$ and $r=2\pi$ with vertical lines.

→ Q2.3.

In [ ]:
u = np.array([1.0, 0.0, 0.0])
r = np.linspace(0, 3 * np.pi, 600)
# TODO: d[i] = dist(p, Exp_p(r[i] u)), err[i] = log_roundtrip_error(S2, p, r[i] u)

# TODO: the two-panel figure described above

### Experiment 2.4: Log is discontinuous at the cut locus
Take two points $q_\pm=\mathrm{Exp}_p\bigl((\pi-\varepsilon)(\pm u)\bigr)$ on opposite sides of the antipode. They are $2\varepsilon$ apart, but their Log vectors point in opposite directions.

In [ ]:
print(f"{'eps':>8} {'d(q+, q-)':>12} {'|Log q+ - Log q-|':>18} {'ratio':>10}")
for eps in [1e-1, 1e-3, 1e-6]:
    # TODO: q_plus, q_minus as above; dq = dist(q_plus, q_minus); dlog = |Log_p q_plus - Log_p q_minus|
    print(f"{eps:8.0e} {dq:12.2e} {dlog:18.4f} {dlog / dq:10.1e}")

### Questions, Part 2
**Q2.1** Using the definition of the injectivity radius in the context, what is the injectivity radius of the unit sphere $S^2$, of a sphere of radius $\rho$, and of the plane $\mathbb R^2$? For the unit sphere, point to where your fan plot shows it.

**Answer Q2.1:**

_Your answer here._

**Q2.2** Why is $\mathrm{Log}_p$ ambiguous at the antipode? Relate your answer to the fan plot.

**Answer Q2.2:**

_Your answer here._

**Q2.3** Does $\mathrm{Exp}_p$ stop existing at the injectivity radius? What do your distance and round-trip plots show?

**Answer Q2.3:**

_Your answer here._

**Q2.4** What happens to statistics (means, covariances, Gaussians) when samples cover a large part of the sphere?

**Answer Q2.4:**

_Your answer here._

**Checkpoint, Part 2.** Public checks pass; fan plot, distance and round-trip curves, sensitivity table; Q2.1 to Q2.4 answered.

---
# Part 3: Orientation representations (18 points)

### Context
**Rotations.** $SO(3)=\{R\in\mathbb R^{3\times3}:R^\top R=I,\ \det R=1\}$ is a 3-dimensional manifold. Every rotation is a rotation by an angle $\theta\in[0,\pi]$ about a unit axis $n$, and the **rotation vector** $\omega=\theta n$ is its logarithm (`rot.rotvec_from_matrix`). The **physical angle between two rotations** is the angle of the relative rotation $R_1^\top R_2$:
$$d(R_1,R_2)=\lVert\log(R_1^\top R_2)\rVert=\arccos\frac{\operatorname{tr}(R_1^\top R_2)-1}{2}.$$
It is *bi-invariant*: $d(AR_1B,AR_2B)=d(R_1,R_2)$ for all rotations $A$ (change of world frame) and $B$ (change of tool frame).

**Unit quaternions.** $q=(w,x,y,z)=\bigl(\cos\tfrac\theta2,\ \sin\tfrac\theta2\,n\bigr)\in S^3$ represents the rotation by θ about $n$. The map $S^3\to SO(3)$ is **2-to-1**: $q$ and $-q$ give the same rotation. $SO(3)$ is $S^3$ with antipodal points identified. Two consequences that you will measure:
* the angle between rotations is twice the $S^3$ distance, taken to the closer of $\pm q_2$: $\ d(R_1,R_2)=2\min\bigl(d_{S^3}(q_1,q_2),\,d_{S^3}(q_1,-q_2)\bigr)$;
* a continuous rotation that turns by 360° moves the quaternion by 180° on $S^3$, from $q$ to $-q$.

Because `UnitQuaternions` is a subclass of your `Sphere(3)`, exp, log, dist and geodesic on $S^3$ come for free. Everything quaternion-specific is about the **sign**.

**Euler angles** are a *chart*: three numbers $(\psi,\vartheta,\varphi)$ (yaw, pitch, roll) per rotation, with $R=R_z(\psi)R_y(\vartheta)R_x(\varphi)$ for SciPy's `"ZYX"`. SciPy returns $\psi,\varphi\in(-180°,180°]$ and $\vartheta\in[-90°,90°]$. No chart of $SO(3)$ is smooth and one-to-one everywhere, and for these Euler angles the trouble sits at pitch $\vartheta=\pm90°$ (gimbal lock):
* **At the pole only a combination of yaw and roll is defined.** At $\vartheta=90°$ the first and the last rotation axis line up, and $R_z(\psi)R_y(90°)R_x(\varphi)=R_z(\psi-\varphi)R_y(90°)$: every pair $(\psi,\varphi)$ with the same difference gives the same rotation, so a library has to pick one pair (SciPy sets one angle to 0). At $\vartheta=-90°$ the same holds for the sum $\psi+\varphi$.
* **Near the pole the angles move fast.** If the tool turns with angular velocity $\omega=(\omega_x,\omega_y,\omega_z)$, expressed in the tool frame, the Euler angles change at the rates
$$\dot\psi=\frac{\sin\varphi\,\omega_y+\cos\varphi\,\omega_z}{\cos\vartheta},\qquad \dot\vartheta=\cos\varphi\,\omega_y-\sin\varphi\,\omega_z,\qquad \dot\varphi=\omega_x+\tan\vartheta\,\bigl(\sin\varphi\,\omega_y+\cos\varphi\,\omega_z\bigr).$$
The factor $1/\cos\vartheta$ means that a slow, smooth rotation can make yaw and roll change very fast when the pitch is close to ±90°.

**Why the screwdriving data is hard for Euler angles.** The screw axis (the tool z-axis) stays within δ of the world x-axis, and the tool spins about it, so $\omega=(0,0,\omega_z)$ in the tool frame. Think about what this spin does to the pitch during one revolution (Experiment 3.2 shows it), and which terms of the rates above it drives. You will explain the failure in Q3.3 and Q3.4.

**Where the quaternion signs come from.** `from_matrix` has to return one of $\pm q$ for each matrix. A rule such as "$w\ge0$" is consistent frame by frame, but along a continuous motion it flips the sign whenever $w$ crosses 0, that is, whenever the rotation angle measured from the identity passes 180°. No rule that looks only at the current matrix can avoid such flips, because a motion that turns by 360° takes a continuously followed $q$ to $-q$ (Q3.1). A sign-continuous trajectory needs the history: that is `align_trajectory`.

### Tasks
* **3.1** `canonical_sign(q)`: flip $q\to-q$ wherever $w<0$.
* **3.2** `align_sign(q_ref, q)`: flip $q$ onto the hemisphere of $q_\text{ref}$, so that $\langle q_\text{ref},q\rangle\ge0$.
* **3.3** `align_trajectory(qs)`: make a trajectory of shape `(T, 4)` sign-continuous by flipping each $q_t$ onto the hemisphere of the already aligned $q_{t-1}$. *Hint:* this can be vectorised with `np.cumprod`.
* **3.4** `rot_dist(q1, q2)`: the physical rotation angle, using your $S^3$ distance.
* **3.5** `matrix_angle(R1, R2)`: the angle from the trace formula, for single matrices or batches `(..., 3, 3)`.
* **3.6** `slerp(q0, q1, t)`: the classical formula $\mathrm{slerp}(q_0,q_1;t)=\frac{\sin((1-t)\Omega)}{\sin\Omega}q_0+\frac{\sin(t\Omega)}{\sin\Omega}q_1$, with $\cos\Omega=\langle q_0,q_1\rangle$, after aligning $q_1$ to $q_0$. Return shape `(len(t), 4)`.

In [ ]:
class UnitQuaternions(Sphere):
    """
    Unit quaternions q = (w, x, y, z) as points of S^3, representing rotations.
    q and -q are the same rotation. exp, log, dist, geodesic, ... are inherited from Sphere(3).
    """

    def __init__(self):
        super().__init__(n=3)

    def __repr__(self):
        return "UnitQuaternions()"

    # ---------------------------------------------------------------- provided
    def from_matrix(self, R):
        """(..., 3, 3) -> (..., 4). The sign is whatever SciPy returns: NOT canonical."""
        return rot.quat_from_matrix(R)

    def to_matrix(self, q):
        return rot.quat_to_matrix(q)

    # ---------------------------------------------------------------- Tasks 3.1 to 3.4, 3.6
    def canonical_sign(self, q):
        """Return q with w >= 0 (flip q -> -q where w < 0)."""
        raise NotImplementedError

    def align_sign(self, q_ref, q):
        """Return +q or -q, whichever has <q_ref, q> >= 0. Broadcasts."""
        raise NotImplementedError

    def align_trajectory(self, qs):
        """(T, 4) -> (T, 4), with <q_t, q_{t-1}> >= 0 for all t and q_0 unchanged."""
        raise NotImplementedError

    def rot_dist(self, q1, q2):
        """Physical rotation angle (rad, in [0, pi]) between the rotations q1 and q2."""
        raise NotImplementedError

    def slerp(self, q0, q1, t):
        """Spherical linear interpolation along the shortest rotation; shape (len(t), 4)."""
        raise NotImplementedError


def matrix_angle(R1, R2):
    """Rotation angle (rad) of R1^T R2 from the trace formula. (..., 3, 3) x (..., 3, 3) -> (...)."""
    raise NotImplementedError


Q = UnitQuaternions()
checks.part3(UnitQuaternions, matrix_angle)

### Using the oracle: geomstats' SO(3) distance
geomstats' metric on $SO(3)$ is defined up to a constant factor, which depends on the version. Find the factor between `SO3.metric.dist` and your `matrix_angle` numerically before you trust it as an oracle.

In [ ]:
# ORACLE
from assignment1utils import oracle
SO3 = oracle.SpecialOrthogonal(n=3, point_type="matrix")
Ra, Rb = Q.to_matrix(Q.random_point(5, rng=rng)), Q.to_matrix(Q.random_point(5, rng=rng))
ratio = SO3.metric.dist(Ra, Rb) / matrix_angle(Ra, Rb)
print("geomstats dist / matrix_angle =", ratio)

### Experiment 3.1: is the data valid?
Report the largest orthonormality error $\lVert R^\top R-I\rVert_\infty$, the range of $\det R$ and the largest $\bigl|\lVert q\rVert-1\bigr|$ over all screwdriving frames.

In [ ]:
Rs_all = screw.poses[..., :3, :3].reshape(-1, 3, 3)
q_all = Q.from_matrix(Rs_all)
# TODO: orth_err, det_min, det_max, qnorm_err
print(f"max |R^T R - I| = {orth_err:.1e},  det R in [{det_min:.15f}, {det_max:.15f}],  max | |q| - 1 | = {qnorm_err:.1e}")

### Experiment 3.2: Euler angles near the pole
**Wrapped and unwrapped angles.** SciPy returns yaw and roll in $(-180°,180°]$. An angle that keeps increasing therefore jumps from +180° to −180° once per turn. That jump comes from the *range* chosen for the angles, not from the singularity of the chart: the angle itself changes continuously, and only its reported value wraps around. `np.unwrap` (which works in radians) removes exactly these jumps, by adding a multiple of 360° whenever two consecutive values differ by more than 180°. Any jump that is still there after unwrapping is a real discontinuity of the chart.

**Plot**
* (a) the ZYX yaw, pitch and roll of the δ = 1° demo against time, in degrees: in one panel as returned by `rot.euler_from_matrix(R, "ZYX")`, in a second panel after `np.unwrap(..., axis=0)`;
* (b) the unwrapped ZYX angles of all three demos, one panel per δ.

Then use the provided `max_euler_step_deg` to compare the largest change of any unwrapped angle between consecutive frames, for `"ZYX"` and for intrinsic `"XYZ"`. The true rotation per frame is 0.576°.

→ Q3.3 and Q3.4.

In [ ]:
t_s = np.arange(screw.poses.shape[1]) * screw.dt
ANGLE_NAMES = ["yaw (Z)", "pitch (Y)", "roll (X)"]

e_raw = rot.euler_from_matrix(screw.poses[1, :, :3, :3], "ZYX")
# TODO (a): two panels for the δ = 1° demo (screw.poses[1]): ZYX angles as returned, and unwrapped

# TODO (b): one panel per demo (δ = 3°, 1°, 0°) with the unwrapped ZYX angles

In [ ]:
def max_euler_step_deg(Rs, seq):
    """Largest change (deg) of any unwrapped Euler angle between consecutive frames."""
    e = np.unwrap(rot.euler_from_matrix(Rs, seq), axis=0)
    return np.degrees(np.abs(np.diff(e, axis=0)).max())

true_step = np.degrees(Q.rot_dist(Q.from_matrix(screw.poses[0, 0, :3, :3]), Q.from_matrix(screw.poses[0, 1, :3, :3])))
print(f"true rotation per frame: {true_step:.3f} deg")
print(f"{'convention':>10} " + " ".join(f"{'δ = ' + format(d, 'g') + '°':>10}" for d in screw.delta_deg))
for seq in ["ZYX", "XYZ"]:
    print(f"{seq:>10} " + " ".join(f"{max_euler_step_deg(T[:, :3, :3], seq):10.3f}" for T in screw.poses))

### Experiment 3.3: the quaternion sign
**Plot** the four components of the quaternions of the δ = 1° demo against time, in three panels: as returned by `from_matrix`, after `canonical_sign`, and after `align_trajectory`. Use `lp.plot_quat_components(q, t_s, ax=ax)`, and mark every full revolution (every 625 frames) with a vertical line.

→ Q3.1.

In [ ]:
T1 = screw.poses[1]
q_raw = Q.from_matrix(T1[:, :3, :3])
# TODO: q_can (canonical signs), q_al (sequentially aligned)

# TODO: the three-panel figure described above

In [ ]:
n_jumps = lambda qq: int(np.sum(np.sum(qq[1:] * qq[:-1], -1) < 0))
print("sign jumps: raw", n_jumps(q_raw), "| canonical", n_jumps(q_can), "| aligned", n_jumps(q_al))
print("aligned q after 1 turn  + q_0:", np.abs(q_al[625] + q_al[0]).max().round(12), "(0 means q_625 = -q_0)")
print("aligned q after 2 turns - q_0:", np.abs(q_al[1250] - q_al[0]).max().round(12), "(0 means q_1250 = +q_0)")

### Experiment 3.4: three ways to measure a small rotation
For consecutive frames of the δ = 1° demo, compare the physical angle, the Euler-difference norm $\lVert\mathrm{wrap}(e_{t+1}-e_t)\rVert$ (with `"ZYX"`), and the quaternion chordal distance $\lVert q_{t+1}-q_t\rVert$ with and without alignment. Write `euler_diff_norm`; you reuse it in Part 5.

Each difference of angles is **wrapped** to $[-180°,180°)$ with `rot.wrap_angle`, for the same reason as the unwrapping in Experiment 3.2: going from +179° to −179° is a step of 2°, not of 358°. Wrapping removes the artefact of the angle range, so that what is left measures the chart.

**Plot** two panels with a logarithmic y-axis, against time:
* left: the physical angle per step and the Euler-difference norm per step, in degrees;
* right: the chordal distance per step, with the signs as returned and after alignment, together with the reference curve $2\sin(\theta/4)$, where θ is the physical angle per step.

→ Q3.3 and Q3.5.

In [ ]:
def euler_diff_norm(R1, R2, seq=rot.EULER_SEQ):
    """| wrap(euler(R2) - euler(R1)) | in rad, each angle difference wrapped to [-pi, pi). Shape (...)."""
    raise NotImplementedError


R_a, R_b = T1[:-1, :3, :3], T1[1:, :3, :3]
# TODO: per-step angle (deg), Euler-difference norm (deg), chordal raw, chordal aligned (each of shape (2500,))

# TODO: the two-panel figure described above (ax.semilogy)

In [ ]:
print(f"physical angle per step: {step_angle.min():.3f}..{step_angle.max():.3f} deg | "
      f"Euler norm: up to {step_euler.max():.1f} deg | chordal raw: up to {step_chord_raw.max():.3f} | "
      f"chordal aligned: up to {step_chord_al.max():.5f}")

### Experiment 3.5: interpolating two orientations
Take frames 1000 and 1300 of the δ = 1° demo (172.8° apart), interpolate between them at the 301 original time stamps with four methods, and measure the angle to the true frames:
1. **linear Euler**: interpolate the `"ZYX"` angles linearly, using the wrapped difference;
2. **SLERP** (your `slerp`);
3. **$S^3$ geodesic** (your `geodesic`) from $q_0$ to the aligned $q_1$, and also to $-q_1$ (the other sign of the same rotation), to see what an unlucky sign does;
4. **matrix geodesic** $R_0\exp\bigl(t\log(R_0^\top R_1)\bigr)$, using `rot.rotvec_from_matrix` and `rot.matrix_from_rotvec`.

**Plot** the angle between each interpolation and the true frames (`matrix_angle`, in degrees) against $t$, one curve per method, with the maximum error of each method in the legend.

→ Q3.5.

In [ ]:
k0, k1 = 1000, 1300
R_true = T1[k0:k1 + 1, :3, :3]
t = np.linspace(0, 1, k1 - k0 + 1)
R0, R1 = R_true[0], R_true[-1]
q0, q1 = Q.from_matrix(R0), Q.from_matrix(R1)
e0, e1 = rot.euler_from_matrix(R0), rot.euler_from_matrix(R1)

interp = {}
# TODO: fill interp[name] = rotation matrices of shape (301, 3, 3) for each method

# TODO: the error of each method against t (one curve per method, maximum error in the legend)

### Deliverable: representation-invariance table
Copy the table into the answer cell below and fill it in with ✓ (invariant) or ✗ (not invariant), then complete `quantities` below. The four columns are the four changes: replacing $q$ by $-q$; switching from `"ZYX"` to `"XYZ"` Euler angles; changing the world frame (multiplying the rotations on the left by a fixed rotation); changing the tool frame (multiplying on the right). The provided harness applies each change to 200 random pairs of rotations and prints the largest deviation, so that you can verify every entry. For the two quantities that are rotations (the midpoint and the mean), "invariant to a frame change" means *equivariant*: changing the frame of the inputs changes the output in the same way.

| Quantity | $q\to-q$ | Euler convention | World frame | Tool frame |
|---|:-:|:-:|:-:|:-:|
| Physical rotation angle between two frames | | | | |
| Geodesic midpoint (as a rotation) | | | | |
| Euler-difference norm | | | | |
| Quaternion chordal distance $\lVert q_1-q_2\rVert$ | | | | |
| Quaternion coordinate mean | | | | |
| Frobenius distance $\lVert R_1-R_2\rVert$ | | | | |

**Answer invariance table:**

_Your answer here._

In [ ]:
def quantities(q1, q2, seq):
    """The six quantities of the table for one pair of rotations q1, q2 (unit quaternions)."""
    R1, R2 = Q.to_matrix(q1), Q.to_matrix(q2)
    # TODO: replace each None (scalars, except the two rotations, which are 3x3 matrices)
    return {
        "physical angle": None,
        "geodesic midpoint (rotation)": None,
        "Euler-difference norm": None,
        "quaternion chordal": None,
        "quaternion coordinate mean (rotation)": None,
        "Frobenius": None,
    }


def invariance_table(n_pairs=200, tol=1e-8, rng=rng):
    """Provided harness: largest deviation of each quantity under each change."""
    qw, qt = Q.random_point(rng=rng), Q.random_point(rng=rng)
    Rw, Rt = Q.to_matrix(qw), Q.to_matrix(qt)
    changes = {
        "q -> -q": (lambda a, b, s: (a, -b, s), lambda M: M),
        "Euler convention": (lambda a, b, s: (a, b, "XYZ"), lambda M: M),
        "world frame": (lambda a, b, s: (rot.quat_multiply(qw, a), rot.quat_multiply(qw, b), s), lambda M: Rw @ M),
        "tool frame": (lambda a, b, s: (rot.quat_multiply(a, qt), rot.quat_multiply(b, qt), s), lambda M: M @ Rt),
    }
    dev = {}
    for _ in range(n_pairs):
        a, b = Q.random_point(rng=rng), Q.random_point(rng=rng)
        base = quantities(a, b, "ZYX")
        for cname, (f_in, f_out) in changes.items():
            new = quantities(*f_in(a, b, "ZYX"))
            for k, v in base.items():
                target = f_out(v) if np.ndim(v) == 2 else v
                dev[k, cname] = max(dev.get((k, cname), 0.0), float(np.abs(new[k] - target).max()))
    print(f"{'quantity':40s}" + "".join(f"{c:>20s}" for c in changes))
    for k in base:
        print(f"{k:40s}" + "".join(f"{('✓' if dev[k, c] < tol else '✗') + f' ({dev[k, c]:.0e})':>20s}" for c in changes))


invariance_table()

### Questions, Part 3
**Q3.1** Why do $q$ and $-q$ represent the same rotation? (Hint: how does a unit quaternion rotate a vector?) Which result of Experiment 3.3 shows that following a rotation continuously through one full revolution takes $q$ to $-q$? Why can `canonical_sign` not remove the jumps, while `align_trajectory` can?

**Answer Q3.1:**

_Your answer here._

**Q3.2** Why can averaging quaternion coordinates fail without sign handling? Use your invariance table.

**Answer Q3.2:**

_Your answer here._

**Q3.3** In Experiment 3.2 the ZYX angles change by much more than the true 0.576° per frame. Explain why, using the context: what does the pitch do during one revolution of the tool, which terms of the Euler rates does the spin $\omega=(0,0,\omega_z)$ drive, and what happens to them as δ shrinks? Why does δ = 0 give jumps of exactly 180°? Quote your numbers from Experiments 3.2 and 3.4.

**Answer Q3.3:**

_Your answer here._

**Q3.4** Intrinsic `"XYZ"`, $R=R_x(a)R_y(b)R_z(c)$, is also singular at $b=\pm90°$, and this motion also passes close to that. Why does it not fail here (largest step 0.576°)? Hint: write the motion as $R(t)=R_\text{start}R_z(\theta(t))$ and ask which of the three XYZ angles a rotation about the tool z-axis changes. How would you change the motion so that XYZ fails and ZYX does not?

**Answer Q3.4:**

_Your answer here._

**Q3.5** Which of your results in this part are intrinsic, and which depend on the representation? Give at least two of each, with the evidence.

**Answer Q3.5:**

_Your answer here._

**Checkpoint, Part 3.** Public checks pass; the figures above; the filled table verified by the harness; Q3.1 to Q3.5 answered.

---
# Part 4: Robot poses and trajectories (18 points)

### Context
A pose is a position $x\in\mathbb R^3$ and an orientation. We represent it as a point of the **product manifold** $\mathbb R^3\times S^3$, stored as a 7-vector $P=(x,q)$, and we treat $q$ and $-q$ as the same orientation. A tangent vector is a 7-vector $V=(u,w)$ with $u\in\mathbb R^3$ and $w\in T_qS^3$.

On a product manifold, geodesics, Exp and Log act **componentwise**:
$$\mathrm{Exp}_{(x,q)}(u,w)=\bigl(x+u,\ \mathrm{Exp}_q w\bigr),\qquad \mathrm{Log}_{(x_0,q_0)}(x_1,q_1)=\bigl(x_1-x_0,\ \mathrm{Log}_{q_0}\tilde q_1\bigr),$$
where $\tilde q_1=\pm q_1$ is aligned with $q_0$ so that Log takes the shorter of the two rotations.

Metres and radians are different units, so the metric needs a weight $\lambda$ (metres per radian):
$$\langle(u_1,w_1),(u_2,w_2)\rangle=u_1^\top u_2+(2\lambda)^2\,w_1^\top w_2,\qquad
d^2\bigl((x_0,q_0),(x_1,q_1)\bigr)=\lVert x_1-x_0\rVert^2+\lambda^2\,d_\text{rot}(q_0,q_1)^2 ,$$
with $d_\text{rot}$ the physical angle from Part 3. The factor $2\lambda$ is there because a rotation by θ is a tangent vector of length θ/2 on $S^3$. With it, $\lVert\mathrm{Log}_PQ\rVert=d(P,Q)$.

**Product geodesics versus screw motions.** The product geodesic treats position and orientation separately: the position moves on a straight line at constant speed while the orientation turns along its own geodesic. This is not the only natural way to go from one pose to another. $SE(3)$ is a group, and its "straight lines" are the **screw motions** $T(t)=T_0\exp\bigl(t\,\xi\bigr)$, with the twist $\xi=\log(T_0^{-1}T_1)$. A twist combines an angular velocity and a linear velocity in the tool frame. By Chasles' theorem, every rigid motion of this form is a rotation about a fixed axis in space at constant speed, combined with a translation along that same axis, like a screw in a nut. The axis does not have to pass through the tool origin: when it does not, the tool origin rotates around the axis and moves on a **helix**, not on a straight line. The screwdriving data was generated as a screw motion (Part 0). Q4.4 asks when the two kinds of interpolation agree.

### Tasks
* **4.1** `PoseManifold`: `inner`, `norm`, `exp`, `log` (with sign alignment), `dist` and `geodesic`. Use your `UnitQuaternions` for the $S^3$ part. The helper `_cat` joins position and quaternion parts with broadcasting.
* **4.2** `interp_pose(T0, T1, t, method)`: interpolate between two 4×4 poses at the times `t` with each of the five methods below, returning `(len(t), 4, 4)`.

In [ ]:
def _cat(x, q):
    """Join (..., 3) and (..., 4) arrays into (..., 7), broadcasting the leading dimensions. Provided."""
    x, q = np.asarray(x, dtype=float), np.asarray(q, dtype=float)
    batch = np.broadcast_shapes(x.shape[:-1], q.shape[:-1])
    return np.concatenate([np.broadcast_to(x, batch + (3,)), np.broadcast_to(q, batch + (4,))], axis=-1)


class PoseManifold:
    """
    Poses as points P = (x, q) of R^3 x S^3, shape (..., 7) = (px, py, pz, qw, qx, qy, qz).
    Tangent vectors V = (u, w), shape (..., 7), with w orthogonal to q.
    Metric: <V1, V2> = u1.u2 + (2 lam)^2 w1.w2, lam in metres per radian.
    q and -q are the same orientation.
    """

    def __init__(self, lam=0.1):
        self.lam = lam
        self.S3 = UnitQuaternions()

    def __repr__(self):
        return f"PoseManifold(lam={self.lam})"

    # ---------------------------------------------------------------- provided
    def from_SE3(self, T):
        """(..., 4, 4) -> (..., 7)."""
        T = np.asarray(T, dtype=float)
        return _cat(T[..., :3, 3], self.S3.from_matrix(T[..., :3, :3]))

    def to_SE3(self, P):
        """(..., 7) -> (..., 4, 4)."""
        P = np.asarray(P, dtype=float)
        q = P[..., 3:] / np.linalg.norm(P[..., 3:], axis=-1, keepdims=True)
        return rot.make_pose(rot.quat_to_matrix(q), P[..., :3])

    def random_point(self, n_samples=None, rng=None, pos_scale=0.1):
        rng = np.random.default_rng() if rng is None else rng
        q = self.S3.random_point(n_samples, rng=rng)
        x = pos_scale * rng.normal(size=q.shape[:-1] + (3,))
        return _cat(x, q)

    # ---------------------------------------------------------------- Task 4.1
    def inner(self, P, U, V):
        raise NotImplementedError

    def norm(self, P, V):
        raise NotImplementedError

    def exp(self, P, V):
        raise NotImplementedError

    def log(self, P, Q):
        raise NotImplementedError

    def dist(self, P, Q):
        raise NotImplementedError

    def geodesic(self, P, Q, t):
        """Points on the product geodesic from P (t=0) to Q (t=1); shape (len(t),) + P.shape."""
        raise NotImplementedError


checks.part4(PoseManifold)

### The five interpolation methods

| Method | Position | Orientation |
|---|---|---|
| `"SLERP"` | linear | your `Q.slerp` |
| `"product geodesic"` | (your `PoseManifold.geodesic`) | (same) |
| `"linear Euler"` | linear | `"ZYX"` angles interpolated linearly, using the wrapped difference |
| `"matrix geodesic"` | linear | $R_0\exp\bigl(t\log(R_0^\top R_1)\bigr)$ |
| `"linear matrix"` | (all 16 entries of $T$ interpolated linearly) | (same) |

In [ ]:
METHODS = ["SLERP", "product geodesic", "linear Euler", "matrix geodesic", "linear matrix"]
POSE = PoseManifold(lam=0.1)


def interp_pose(T0, T1, t, method):
    """Interpolate between the poses T0 and T1 (4x4) at times t (shape (n,)); returns (n, 4, 4)."""
    t = np.asarray(t, dtype=float)
    R0, R1 = T0[:3, :3], T1[:3, :3]
    x_lin = (1 - t)[:, None] * T0[:3, 3] + t[:, None] * T1[:3, 3]
    if method == "SLERP":
        return 
    if method == "product geodesic":
        return 
    if method == "linear Euler":
        return 
    if method == "matrix geodesic":
        return 
    if method == "linear matrix":
        return 
    raise ValueError(method)
    

### Experiment 4.1: keyframe interpolation with a known answer
Keep every $k$-th frame of a demo (and the last frame) as a keyframe, interpolate between consecutive keyframes at the original time stamps, and compare with every true frame in between. Because the motion is a constant twist about a fixed axis, the true path between two keyframes less than 180° apart is exactly the product geodesic. For `"linear matrix"` the interpolated "rotation" is not a rotation; the harness measures its angle after projecting it to $SO(3)$, and reports its determinant separately.

The harness `keyframe_errors` and the table are provided. **Plot** $\det R$ along the `"linear matrix"` interpolation of one segment, from keyframe 900 to keyframe 1200 of the δ = 1° demo (172.8° of rotation), against $t$.

→ Q4.3 (table) and Q4.5 (det plot).

In [ ]:
def keyframe_errors(T, k, method):
    """Max rotation error (deg) and max position error (mm) when interpolating keyframes every k frames. Provided."""
    keys = np.unique(np.r_[np.arange(0, len(T), k), len(T) - 1])
    rot_err, pos_err = 0.0, 0.0
    for a, b in zip(keys[:-1], keys[1:]):
        Ti = interp_pose(T[a], T[b], np.linspace(0, 1, b - a + 1), method)
        rot_err = max(rot_err, np.degrees(matrix_angle(rot.project_SO3(Ti[:, :3, :3]), T[a:b + 1, :3, :3])).max())
        pos_err = max(pos_err, 1000 * np.linalg.norm(Ti[:, :3, 3] - T[a:b + 1, :3, 3], axis=-1).max())
    return rot_err, pos_err


import pandas as pd

SPACINGS = [125, 250, 300, 350]
rows = []
for T, d in zip(screw.poses, screw.delta_deg):
    for k in SPACINGS:
        for m in METHODS:
            re, pe = keyframe_errors(T, k, m)
            rows.append(dict(delta=f"{d:g}°", k=k, rotation_between_keys=f"{k * 0.576:.1f}°", method=m, max_rot_err_deg=re, max_pos_err_mm=pe))
kf = pd.DataFrame(rows)
table = kf.pivot_table(index=["delta", "k", "rotation_between_keys"], columns="method", values="max_rot_err_deg", sort=False)[METHODS]
print("Max rotation error [deg] when interpolating keyframes:")
table.round(2)

In [ ]:
print(f"largest position error over all runs: {kf.max_pos_err_mm.max():.2e} mm")

T = screw.poses[1]
Ti = interp_pose(T[900], T[1200], np.linspace(0, 1, 301), "linear matrix")
Rlin = Ti[:, :3, :3]
# TODO: plot det(Rlin) against t = np.linspace(0, 1, 301)
print(f"smallest det R along the segment: {np.linalg.det(Rlin).min():.4f}")

### One segment in 3D
The figure shows the segment from keyframe 900 to 1200 (δ = 1°, 172.8° of rotation) for three methods. The true path is shown in every panel as thin grey frames.

In [ ]:
T = screw.poses[1]
seg = T[900:1201]
fig = plt.figure(figsize=(16, 5))
for j, m in enumerate(["SLERP", "linear Euler", "linear matrix"]):
    ax = fig.add_subplot(1, 3, j + 1, projection="3d")
    Ti = interp_pose(T[900], T[1200], np.linspace(0, 1, 301), m)
    idx = lp.subsample_idx(301, 12)
    for i in idx:
        for a in range(3):
            ax.plot(*np.stack([seg[i, :3, 3], seg[i, :3, 3] + 0.0012 * seg[i, :3, a]]).T, color="0.75", lw=2.5)
    lp.plot_frames_3d(Ti, ax=ax, n_frames=12, frame_len=0.0012, colors=[lp.METHOD_COLORS.get(m, "k")], title=m)
plt.tight_layout(); plt.show()

### Experiment 4.2: the role of λ
(a) Check that the product geodesic does not change with λ. (b) Take a target pose $P_0$ and two candidate poses: $A$ is 5 cm away with the same orientation, and $B$ is at the same position but rotated by 10°. For which values of λ is $A$ closer than $B$? **Plot** the distances from $P_0$ to $A$ and to $B$ against λ on log–log axes, and mark the crossover value λ* with a vertical line.

→ Q4.1 and Q4.2.

In [ ]:
P0 = POSE.from_SE3(screw.poses[1, 0])
P1 = POSE.from_SE3(screw.poses[1, 300])
tt = np.linspace(0, 1, 11)
geo = {lam: PoseManifold(lam).geodesic(P0, P1, tt) for lam in [0.01, 0.1, 1.0]}
print("max difference between geodesics for lam = 0.01 and 1:", np.abs(geo[0.01] - geo[1.0]).max())

A = POSE.exp(P0, np.r_[0.05, 0, 0, 0, 0, 0, 0])            # 5 cm away, same orientation
w = Q.to_tangent(P0[3:], rng.normal(size=4))
w *= 0.5 * np.radians(10) / np.linalg.norm(w)              # |w| = theta / 2 on S^3
B = POSE.exp(P0, np.r_[0, 0, 0, w])                         # same position, rotated by 10 deg

lams = np.logspace(-2, 1, 200)
# TODO: dA[i], dB[i] = distances from P0 to A and to B for lam = lams[i]; lam_star = crossover
# TODO: the figure described above
print(f"crossover: lam* = {lam_star:.4f} m/rad")

### Questions, Part 4
**Q4.1** What are the units of λ? How would you choose it for a screwdriver? Use your crossover value λ* from Experiment 4.2.

**Answer Q4.1:**

_Your answer here._

**Q4.2** Why does the product geodesic not depend on λ? Which quantities do depend on it?

**Answer Q4.2:**

_Your answer here._

**Q4.3** Why do all intrinsic methods fail at a spacing of 350 frames (201.6°)? Is that a failure of the geometry or of the data?

**Answer Q4.3:**

_Your answer here._

**Q4.4** Is the product geodesic the same as the $SE(3)$ screw motion $T_0\exp(t\log(T_0^{-1}T_1))$ in general? Why do they coincide here?

**Answer Q4.4:**

_Your answer here._

**Q4.5** What happens to $\det R$ along the linear interpolation of the 4×4 matrices in your plot, and why? What does a rotation matrix with determinant close to 0 do to the tool frame (look at the 3D figure)? Why is $\mathbb R^{16}$ the wrong space in which to interpolate poses?

**Answer Q4.5:**

_Your answer here._

**Checkpoint, Part 4.** Public checks pass; the error table for all methods, spacings and δ; the det plot; the 3D figure; the λ figure; Q4.1 to Q4.5 answered.

---
# Part 5: Tracking-error metrics (13 points)

### Context
To evaluate a controller or a learned policy we compare an executed trajectory with a reference, frame by frame. For positions the choice is obvious. For orientations there are many candidates:

| Metric | Formula | Unit |
|---|---|---|
| position error | $\lVert p-p_\text{ref}\rVert$ | mm |
| rotation angle (intrinsic) | $\arccos\frac{\operatorname{tr}(R_\text{ref}^\top R)-1}2$ | deg |
| Euler-difference norm | $\lVert\mathrm{wrap}(e(R)-e(R_\text{ref}))\rVert$, ZYX | deg |
| quaternion chordal distance | $\lVert q-q_\text{ref}\rVert$, with or without sign alignment | – |
| Frobenius distance | $\lVert R-R_\text{ref}\rVert_F$ | – |
| weighted pose error | $\sqrt{\lVert p-p_\text{ref}\rVert^2+\lambda^2\theta^2}$ | m |

A good metric should reflect the **motion**, not the **representation**, and you should know which part of it is a **choice** (the metric and λ).

### Tasks
* **5.1** Write the six metric functions below. Each takes two arrays of poses `(N, 4, 4)` (reference and executed) and returns `(N,)`. Reuse `matrix_angle`, `euler_diff_norm` and `Q.align_sign`.

In [ ]:
def position_error_mm(T_ref, T):
    return 1000.0 * np.linalg.norm(T[..., :3, 3] - T_ref[..., :3, 3], axis=-1)


def rotation_angle_deg(T_ref, T):
    raise NotImplementedError


def euler_error_deg(T_ref, T, seq=rot.EULER_SEQ):
    raise NotImplementedError


def quat_chordal(T_ref, T, aligned=True):
    """| q - q_ref | with q from Q.from_matrix; if aligned, q is first moved to the hemisphere of q_ref."""
    raise NotImplementedError


def frobenius_error(T_ref, T):
    raise NotImplementedError


def pose_error(T_ref, T, lam=0.1):
    """sqrt(|dp|^2 + lam^2 theta^2) in metres, theta in radians, lam in m/rad."""
    raise NotImplementedError


METRICS = dict(position_error_mm=position_error_mm, rotation_angle_deg=rotation_angle_deg,
               euler_error_deg=euler_error_deg, quat_chordal=quat_chordal,
               frobenius_error=frobenius_error, pose_error=pose_error)
checks.part5(METRICS)

### Experiment 5.A: a controlled error
In `screw_tracked`, every frame of the executed trajectory is the reference frame followed by the same tool-frame offset: 1° and 2 mm. The true error is therefore constant. `all_metrics` (provided) returns every metric as a dict of arrays. **Plot** every metric against time, one panel per metric (7 panels sharing the time axis), with one curve per demo (δ = 3°, 1°, 0°). Start each y-axis at 0, so that rounding noise of $10^{-12}$ is not magnified.

→ Q5.1 and Q5.2.

In [ ]:
trk = data.load_screw_tracked()
print("offset angle:", np.degrees(np.linalg.norm(trk.offset_rotvec)).round(6), "deg; offset distance:",
      (1000 * np.linalg.norm(trk.offset_pos)).round(6), "mm")


def all_metrics(T_ref, T, lam=0.1):
    """Every metric as a dict of (N,) arrays. Provided."""
    return {
        "position [mm]": position_error_mm(T_ref, T),
        "angle [deg]": rotation_angle_deg(T_ref, T),
        "Euler ZYX [deg]": euler_error_deg(T_ref, T),
        "chordal, raw": quat_chordal(T_ref, T, aligned=False),
        "chordal, aligned": quat_chordal(T_ref, T, aligned=True),
        "Frobenius": frobenius_error(T_ref, T),
        f"pose, λ={lam} [m]": pose_error(T_ref, T, lam),
    }


names = list(all_metrics(trk.reference[0, :2], trk.tracked[0, :2]))
# TODO: one panel per metric (names), one curve per demo, against t_s; y-axes starting at 0

In [ ]:
for i, d in enumerate(trk.delta_deg):
    M = all_metrics(trk.reference[i], trk.tracked[i])
    print(f"δ = {d:g}°: " + ", ".join(f"{n}: {M[n].min():.3g}..{M[n].max():.3g}" for n in names[1:5]))

### Experiment 5.B: real data, water versus wine
**The pouring data.** `data.load_pouring()` contains the pose of a bottle held by the robot while it pours, 480 frames per demonstration. All demonstrations start from almost the same pose and move to one of 5 target positions, from very left to very right; there the bottle tilts to pour and then comes back up. For each position there is one demonstration pouring water and one pouring wine. They have the same structure but were recorded separately, so they differ by several centimetres (4 to 13 cm at the end point) and by several degrees. The sampling rate is not recorded, so we use the frame index as time. The provided figure shows the bottle paths with a few bottle frames, and the water and wine paths on top of each other.

In [ ]:
pour = data.load_pouring()
cols = plt.cm.viridis(np.linspace(0, 0.9, 5))
fig = plt.figure(figsize=(17, 5.5))
for j, (name, demos) in enumerate([("water", pour.water), ("wine", pour.wine)]):
    ax = fig.add_subplot(1, 3, j + 1, projection="3d")
    lp.plot_frames_3d(list(demos), ax=ax, labels=pour.position_names, colors=cols, n_frames=8,
                      title=f"{name}: bottle path and frames")
ax = fig.add_subplot(1, 3, 3, projection="3d")
for k in range(5):
    ax.plot(*pour.water[k][:, :3, 3].T, color=cols[k], lw=1.5)
    ax.plot(*pour.wine[k][:, :3, 3].T, color=cols[k], lw=1.5, ls="--")
lp.set_equal_3d(ax, pour.water[:, :, :3, 3])
ax.set_title("water (solid) vs wine (dashed)", fontsize=10)
plt.tight_layout(); plt.show()

print("distance between the water and wine end positions [cm]:",
      (100 * np.linalg.norm(pour.water[:, -1, :3, 3] - pour.wine[:, -1, :3, 3], axis=-1)).round(1))

Treat each wine demo as an *execution* of the water demo with the same target position, frame by frame (both have 480 frames). **Plot** six panels, one per metric (position, angle, Euler ZYX, chordal raw, chordal aligned, Frobenius), each with one curve per position against the frame index. Then compute the Spearman rank correlation between the metrics over all frames (the correlation plot is provided), and rank the five positions by their mean error under each metric and for λ ∈ {0.01, 0.1, 1} m/rad.

→ Q5.1, Q5.2 and Q5.3.

In [ ]:
from scipy.stats import spearmanr

pour = data.load_pouring()
per_pos = [all_metrics(pour.water[k], pour.wine[k]) for k in range(5)]

# TODO: the six panels described above (one curve per position, label with pour.position_names)

In [ ]:
# TODO: rho = Spearman correlation matrix (len(names) x len(names)) over all 5 x 480 frames
lp.plot_corr_matrix(rho, names, title="Spearman rank correlation between metrics (pouring)")
plt.show()

In [ ]:
rank_rows = {}
for n in names[:-1]:
    means = np.array([per_pos[k][n].mean() for k in range(5)])
    rank_rows[n] = means
for lam in [0.01, 0.1, 1.0]:
    # TODO: mean pose error for each position with this lam
    pass
means = pd.DataFrame(rank_rows, index=pour.position_names).T
ranks = means.rank(axis=1).astype(int)
print("Mean error per position:"); display(means.round(3))
print("Rank of each position (1 = smallest mean error):"); display(ranks)

### Questions, Part 5
**Q5.1** For each metric, say whether the differences you observed in Experiment 5.A and 5.B reflect the motion, the representation or the choice of metric.

**Answer Q5.1:**

_Your answer here._

**Q5.2** The Frobenius distance and the aligned chordal distance are monotone functions of the rotation angle θ. Derive both formulas and check them against your numbers from Experiment 5.A. When does the difference between them and θ still matter?

*Hints.* The Frobenius norm of a matrix is $\lVert A\rVert_F=\sqrt{\sum_{ij}A_{ij}^2}=\sqrt{\operatorname{tr}(A^\top A)}$. Expand $\lVert R_1-R_2\rVert_F^2$ using $R^\top R=I$, and use $\operatorname{tr}(R_1^\top R_2)=1+2\cos\theta$ (the trace formula of Part 3). For quaternions, expand $\lVert q_1-q_2\rVert^2$ using $\lVert q\rVert=1$, and use $\langle q_1,q_2\rangle=\cos(\theta/2)$ after alignment. Finish with $1-\cos x=2\sin^2(x/2)$.

**Answer Q5.2:**

_Your answer here._

**Q5.3** Which λ would you use for pouring, and does the ranking of the five positions depend on it?

**Answer Q5.3:**

_Your answer here._

**Q5.4** Which error metric would you report in a paper on pose tracking, and why?

**Answer Q5.4:**

_Your answer here._

**Checkpoint, Part 5.** Public checks pass; the two multi-panel figures, the correlation matrix, the ranking table; Q5.1 to Q5.4 answered.

---
# Final synthesis (16 points)

S.1, S.2, S.4 and S.5 are worth 3 points each, S.3 and S.6 2 points each. Answer each in a short paragraph and support it with results from your own run.

**S.1** Pick one result from this assignment that is **intrinsic** and one that is **representation-dependent**. For each, give the number and say what would change it.

**Answer S.1:**

_Your answer here._

**S.2** A colleague trains a network to output end-effector orientations for the screwdriving task and asks you which output representation and which loss to use. Give a recommendation in three or four sentences, based on your results.

**Answer S.2:**

_Your answer here._

**S.3** You record a robot turning a valve handle. The orientation of the gripper is logged at 5 Hz while the handle turns at up to 3 revolutions per second, and you need a smooth trajectory between the logged samples. Which tools of this assignment do you use, and what do you check in the data before trusting the result?

**Answer S.3:**

_Your answer here._

**S.4** Name two results of this assignment that depend on a *choice of metric*, not on the representation and not on the motion. For each, give your number and say how you would report the result so that a reader can reproduce your conclusion.

**Answer S.4:**

_Your answer here._

**S.5** Curvature changed the outcome of several experiments. Choose two, and for each say what the same computation gives in flat space and what you measured on the sphere.

**Answer S.5:**

_Your answer here._

**S.6** A dataset stores end-effector orientations as unit quaternions. List, in order, the processing steps you would apply before computing tracking errors against a reference, and justify each step with a result of this assignment.

**Answer S.6:**

_Your answer here._